In [11]:
# =============================================
# 1. 더미 뉴스 데이터 생성
# =============================================

dummy_news = [
    {"text": "삼성전자, 반도체 투자 확대 발표", "stocks": ["삼성전자"]},
    {"text": "SK하이닉스, HBM 수요 증가 소식", "stocks": ["SK하이닉스"]},
    {"text": "LG화학, 배터리 신기술 개발", "stocks": ["LG화학"]},
    {"text": "삼성전자와 SK하이닉스, 공급망 협력 강화", "stocks": ["삼성전자", "SK하이닉스"]},
    {"text": "LG화학과 삼성전자 소재 협약 체결", "stocks": ["LG화학", "삼성전자"]},
]

tickers = ["삼성전자", "SK하이닉스", "LG화학"]


In [12]:
# =============================================
# 2. 더미 임베딩 함수
# =============================================
import torch
import numpy as np

def dummy_embedding(text):
    # 텍스트 길이를 반영해도 되고, 지금은 임시로 랜덤 8차원 벡터
    return torch.tensor(np.random.rand(8), dtype=torch.float32)


In [13]:
# =============================================
# 3. 종목별 node feature 생성 (뉴스 임베딩 평균)
# =============================================

node_features = {t: [] for t in tickers}

for news in dummy_news:
    emb = dummy_embedding(news["text"])
    for stock in news["stocks"]:
        node_features[stock].append(emb)

# 평균 pooling
for stock in node_features:
    node_features[stock] = torch.stack(node_features[stock]).mean(dim=0)

node_features


{'삼성전자': tensor([0.5657, 0.6014, 0.5114, 0.4633, 0.3170, 0.7184, 0.6672, 0.3372]),
 'SK하이닉스': tensor([0.7357, 0.6162, 0.5773, 0.7939, 0.4385, 0.8409, 0.8180, 0.3804]),
 'LG화학': tensor([0.6558, 0.5980, 0.3068, 0.5644, 0.1984, 0.4780, 0.6665, 0.5522])}

In [14]:
# =============================================
# 4. edge_index 생성 (공동 언급 뉴스 기반)
# =============================================

edges = []

for news in dummy_news:
    stocks = news["stocks"]
    if len(stocks) > 1:
        for i in range(len(stocks)):
            for j in range(i+1, len(stocks)):
                # undirected edge
                edges.append((stocks[i], stocks[j]))
                edges.append((stocks[j], stocks[i]))

stock_to_id = {name: idx for idx, name in enumerate(tickers)}

edge_index = torch.tensor([
    [stock_to_id[s] for s, t in edges],
    [stock_to_id[t] for s, t in edges]
], dtype=torch.long)

edge_index


tensor([[0, 1, 2, 0],
        [1, 0, 0, 2]])

In [8]:
from torch_geometric.nn import GCNConv

class MiniGCN(torch.nn.Module):
    def __init__(self, in_dim=8, hid_dim=16, out_dim=8):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hid_dim)
        self.conv2 = GCNConv(hid_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = MiniGCN()

X = torch.stack([node_features[t] for t in tickers])  # shape [3, 8]






In [9]:
output = model(X, edge_index)
print(output)



RuntimeError: expected m1 and m2 to have the same dtype, but got: double != float

In [10]:
import torch.nn.functional as F

cos = F.cosine_similarity(output[0], output[1], dim=0)
print("삼성전자 vs SK하이닉스 유사도:", cos.item())


NameError: name 'output' is not defined